In [ ]:
# Instalação das Bibliotecas

# pip install selenium
# pip install tinydb
# pip install fake_useragent

### Criação do WebScraping

In [1]:
import pandas as pd
from time import sleep
from datetime import date, datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from tqdm.auto import tqdm
from tinydb import TinyDB, Query
from bs4 import BeautifulSoup
from fake_useragent import UserAgent

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
leagues_list = ["ARGENTINA - COPA DE LA LIGA PROFESIONAL",
"ARGENTINA - LIGA PROFESIONAL", 
"ARMENIA - PREMIER LEAGUE", 
"AUSTRALIA - A-LEAGUE",
"AUSTRIA - 2. LIGA", 
"AUSTRIA - BUNDESLIGA",
"BELGIUM - CHALLENGER PRO LEAGUE", 
"BELGIUM - JUPILER PRO LEAGUE",
"BOLIVIA - DIVISION PROFESIONAL", 
"BOSNIA AND HERZEGOVINA - PREMIJER LIGA BIH",
"BRAZIL - SERIE A", 
"BRAZIL - SERIE B", 
"BRAZIL - SERIE C",
"BRAZIL - SERIE D", 
"BULGARIA - PARVA LIGA",
"CHILE - PRIMERA DIVISION",
"CHINA - SUPER LEAGUE", 
"COLOMBIA - PRIMERA A",
"CROATIA - HNL", 
"CROATIA - PRVA NL", 
"CYPRUS - CYTA CHAMPIONSHIP",
"CZECH REPUBLIC - FORTUNA:LIGA", 
"DENMARK - 1ST DIVISION",
"DENMARK - SUPERLIGA", 
"ECUADOR - LIGA PRO", 
"EGYPT - PREMIER LEAGUE", 
"ENGLAND - CHAMPIONSHIP",
"ENGLAND - LEAGUE ONE",
"ENGLAND - LEAGUE TWO", 
"ENGLAND - NATIONAL LEAGUE",
"ENGLAND - PREMIER LEAGUE", 
"ESTONIA - ESILIIGA", 
"ESTONIA - MEISTRILIIGA",
"EUROPE - CHAMPIONS LEAGUE",
"EUROPE - EUROPA CONFERENCE LEAGUE", 
"EUROPE - EUROPA LEAGUE",
"FINLAND - VEIKKAUSLIIGA",
"FINLAND - YKKONEN", 
"FRANCE - LIGUE 1", 
"FRANCE - LIGUE 2",
"FRANCE - NATIONAL", 
"GERMANY - 2. BUNDESLIGA", 
"GERMANY - 3. LIGA",
"GERMANY - BUNDESLIGA", 
"GREECE - SUPER LEAGUE",
"HUNGARY - OTP BANK LIGA",
"ICELAND - BESTA DEILD KARLA", 
"IRELAND - DIVISION 1",
"IRELAND - PREMIER DIVISION", 
"ISRAEL - LIGAT HA'AL",
"ITALY - SERIE A", 
"ITALY - SERIE B",
"ITALY - SERIE C", 
"ITALY - SERIE D", 
"JAPAN - J1 LEAGUE",
"JAPAN - J2 LEAGUE",  
"MEXICO - LIGA MX",
"NETHERLANDS - EERSTE DIVISIE", 
"NETHERLANDS - EREDIVISIE",
"NORTHERN IRELAND - NIFL CHAMPIONSHIP",
"NORTHERN IRELAND - NIFL PREMIERSHIP", 
"NORWAY - ELITESERIEN",
"NORWAY - OBOS-LIGAEN", 
"PARAGUAY - PRIMERA DIVISION",
"PERU - LIGA 1", 
"POLAND - DIVISION 1", 
"POLAND - EKSTRAKLASA",
"PORTUGAL - LIGA PORTUGAL", 
"PORTUGAL - LIGA PORTUGAL 2",
"ROMANIA - LIGA 1",
"ROMANIA - LIGA 2",
"SAUDI ARABIA - SAUDI PROFESSIONAL LEAGUE",
"SCOTLAND - CHAMPIONSHIP", 
"SCOTLAND - LEAGUE ONE",
"SCOTLAND - LEAGUE TWO", 
"SCOTLAND - PREMIERSHIP",
"SERBIA - PRVA LIGA",
"SERBIA - SUPER LIGA",
"SLOVAKIA - NIKE LIGA",
"SLOVENIA - PRVA LIGA", 
"SOUTH AFRICA - PREMIER LEAGUE",
"SOUTH AMERICA - COPA LIBERTADORES",
"SOUTH AMERICA - COPA SUDAMERICANA", 
"SOUTH KOREA - K LEAGUE 1",
"SOUTH KOREA - K LEAGUE 2", 
"SPAIN - LALIGA", 
"SPAIN - LALIGA2", 
"SPAIN - PRIMERA RFEF",
"SPAIN - SEGUNDA RFEF", 
"SWEDEN - ALLSVENSKAN", 
"SWEDEN - SUPERETTAN",
"SWITZERLAND - CHALLENGE LEAGUE", 
"SWITZERLAND - SUPER LEAGUE",
"TURKEY - 1. LIG",
"TURKEY - SUPER LIG", 
"UKRAINE - PREMIER LEAGUE", 
"URUGUAY - PRIMERA DIVISION",
"USA - MLS", 
"VENEZUELA - LIGA FUTVE", 
"WALES - CYMRU PREMIER"]

In [ ]:
def get_main_header(id_jogo,dictionary,driver):
    driver.get(f'https://www.flashscore.com/match/{id_jogo}/#/match-summary/match-summary')
    dictionary['Id'] = id_jogo
    country = driver.find_element(By.CSS_SELECTOR,'span.tournamentHeader__country').text.split(':')[0]
    date = driver.find_element(By.CSS_SELECTOR,'div.duelParticipant__startTime').text.split(' ')[0]
    dictionary['Date'] = date.replace('.','/')
    time = driver.find_element(By.CSS_SELECTOR,'div.duelParticipant__startTime').text.split(' ')[1]
    dictionary['Time'] = time
    league = driver.find_element(By.CSS_SELECTOR,'span.tournamentHeader__country > a').text.split(' -')[0]
    dictionary['League'] = f'{country} - {league}'
    home = driver.find_element(By.CSS_SELECTOR,'div.duelParticipant__home').find_element(By.CSS_SELECTOR,'div.participant__participantName').text
    dictionary['Home'] = home
    away = driver.find_element(By.CSS_SELECTOR,'div.duelParticipant__away').find_element(By.CSS_SELECTOR,'div.participant__participantName').text
    dictionary['Away'] = away
    try:
        rodada = driver.find_element(By.CSS_SELECTOR,'span.tournamentHeader__country > a').text.split('- ')[1]
        dictionary['Round'] = rodada
    except:
        dictionary['Round'] = '-'

In [ ]:
def get_odds_ml_ft(id_jogo,dictionary,driver):
    url_ml_full_time = f'https://www.flashscore.com/match/{id_jogo}/#/odds-comparison/1x2-odds/full-time'
    driver.get(url_ml_full_time)
    sleep(1)
    if driver.current_url == url_ml_full_time:
        WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.ui-table')))
        table_odds = driver.find_element(By.CSS_SELECTOR,'div.ui-table')
        linha_ml_ft = table_odds.find_element(By.CSS_SELECTOR,'div.ui-table__row')
        dictionary['FT_Odd_H'] = float(linha_ml_ft.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
        dictionary['FT_Odd_D'] = float(linha_ml_ft.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)
        dictionary['FT_Odd_A'] = float(linha_ml_ft.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[2].text)

In [ ]:
def get_odds_ou_ht(id_jogo,dictionary,driver):
    url_ou_first_half = f'https://www.flashscore.com/match/{id_jogo}/#/odds-comparison/over-under/1st-half'
    driver.get(url_ou_first_half)
    sleep(1)
    if driver.current_url == url_ou_first_half:
        WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.ui-table')))
        linhas = driver.find_elements(By.CSS_SELECTOR,'div.ui-table__body')
        for linha in linhas:
            if (len(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')) > 1):
                bookie = linha.find_element(By.CSS_SELECTOR,'img.prematchLogo').get_attribute('title')
                total_gols = linha.find_element(By.CSS_SELECTOR,'span.oddsCell__noOddsCell').text.replace('.','')
                if total_gols == '05':
                    over = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
                    under = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)
                    dictionary[f'HT_Odd_Over{total_gols}'] = over
                    dictionary[f'HT_Odd_Under{total_gols}'] = under
                    del bookie,total_gols,over,under

In [ ]:
def get_odds_ou_ft(id_jogo,dictionary,driver):
    url_ou_full_time = f'https://www.flashscore.com/match/{id_jogo}/#/odds-comparison/over-under/full-time'
    driver.get(url_ou_full_time)
    sleep(1)
    if driver.current_url == url_ou_full_time:
        WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.ui-table')))
        linhas = driver.find_elements(By.CSS_SELECTOR,'div.ui-table__body')
        for linha in linhas:
            if (len(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')) > 1):
                bookie = linha.find_element(By.CSS_SELECTOR,'img.prematchLogo').get_attribute('title')
                total_gols = linha.find_element(By.CSS_SELECTOR,'span.oddsCell__noOddsCell').text.replace('.','')
                if total_gols == '05':
                    over = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
                    under = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)
                    dictionary[f'FT_Odd_Over{total_gols}'] = over
                    dictionary[f'FT_Odd_Under{total_gols}'] = under
                    del bookie,total_gols,over,under
                elif total_gols == '15':
                    over = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
                    under = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)
                    dictionary[f'FT_Odd_Over{total_gols}'] = over
                    dictionary[f'FT_Odd_Under{total_gols}'] = under
                    del bookie,total_gols,over,under
                elif total_gols == '25':
                    over = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
                    under = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)
                    dictionary[f'FT_Odd_Over{total_gols}'] = over
                    dictionary[f'FT_Odd_Under{total_gols}'] = under
                    del bookie,total_gols,over,under

In [ ]:
def get_odds_btts_ft(id_jogo,dictionary,driver):
    url_btts_full_time = f'https://www.flashscore.com/match/{id_jogo}/#/odds-comparison/both-teams-to-score/full-time'
    driver.get(url_btts_full_time)
    sleep(1)
    if driver.current_url == url_btts_full_time:
        WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.ui-table')))
        linha = driver.find_element(By.CSS_SELECTOR,'div.ui-table__row')
        bookie_btts = linha.find_element(By.CSS_SELECTOR,'img.prematchLogo').get_attribute('title')
        dictionary['Odd_BTTS_Yes'] = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
        dictionary['Odd_BTTS_No'] = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)

In [ ]:
def get_odds_ah_ft(id_jogo,dictionary,driver):
    url_ah_full_time = f'https://www.flashscore.com/match/{id_jogo}/#/odds-comparison/asian-handicap/full-time'
    driver.get(url_ah_full_time)
    sleep(1)
    if driver.current_url == url_ah_full_time:
        WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.ui-table')))
        linhas = driver.find_elements(By.CSS_SELECTOR,'div.ui-table__body')
        for linha in linhas:
            if (len(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')) > 1):
                bookie = linha.find_element(By.CSS_SELECTOR,'img.prematchLogo').get_attribute('title')
                linha_handcap = linha.find_element(By.CSS_SELECTOR,'span.oddsCell__noOddsCell').text
                if (linha_handcap == '-0.5'):
                    linha_handcap = linha_handcap.replace('-','')
                    linha_handcap = linha_handcap.replace('.','')
                    ah_home = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
                    ah_away = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)
                    dictionary[f'Odd_AH_Neg{linha_handcap}_H'] = ah_home
                    dictionary[f'Odd_AH_Pos{linha_handcap}_A'] = ah_away
                    del bookie,linha_handcap,ah_home,ah_away
                elif (linha_handcap == '+0.5'):
                    linha_handcap = linha_handcap.replace('+','')
                    linha_handcap = linha_handcap.replace('.','')
                    ah_home = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[0].text)
                    ah_away = float(linha.find_elements(By.CSS_SELECTOR,'a.oddsCell__odd')[1].text)
                    dictionary[f'Odd_AH_Pos{linha_handcap}_H'] = ah_home
                    dictionary[f'Odd_AH_Neg{linha_handcap}_A'] = ah_away
                    del bookie,linha_handcap,ah_home,ah_away

In [ ]:
def get_odds_cs_ft(id_jogo,dictionary,driver):
    url_cs_full_time = f'https://www.flashscore.com/match/{id_jogo}/#/odds-comparison/correct-score/full-time'
    driver.get(url_cs_full_time)
    sleep(1)
    if driver.current_url == url_cs_full_time:
        WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.ui-table')))
        for celula in driver.find_elements(By.CSS_SELECTOR,'div.ui-table__body'):
            linha = celula.find_element(By.CSS_SELECTOR,'div.ui-table__row')
            bookie = linha.find_element(By.CSS_SELECTOR,'img.prematchLogo').get_attribute('title')
            placar = linha.find_element(By.CSS_SELECTOR,'span.oddsCell__noOddsCell').text.replace(':','-')
            if (placar == '0-0'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '0-1'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '0-2'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '0-3'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '1-0'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '1-1'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '1-2'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '1-3'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '2-0'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '2-1'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '2-2'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '2-3'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '3-0'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '3-1'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '3-2'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '3-3'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd
            elif (placar == '4-4'):
                odd = float(linha.find_element(By.CSS_SELECTOR,'a.oddsCell__odd').text)
                dictionary[f'Odd_CS_{placar}'] = odd
                del linha,bookie,placar,odd

In [ ]:
def insert_season(dictionary,season):
    dictionary['Season'] = season

In [ ]:
def get_score_ht_ft(id_jogo,dictionary,driver):
    try:
        if '1st Half' in driver.page_source:
            WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.smv__incidentsHeader')))
            home_goals_ht = int(driver.find_elements(By.CSS_SELECTOR,'div.smv__incidentsHeader div')[1].text.split('-')[0])
            dictionary['HT_Goals_H'] = home_goals_ht
            away_goals_ht = int(driver.find_elements(By.CSS_SELECTOR,'div.smv__incidentsHeader div')[1].text.split('-')[1])
            dictionary['HT_Goals_A'] = away_goals_ht
    except:
        pass
    home_goals_ft = driver.find_element(By.CSS_SELECTOR,'div.detailScore__wrapper').text.split('-')[0]
    dictionary['FT_Goals_H'] = int(home_goals_ft)
    away_goals_ft = driver.find_element(By.CSS_SELECTOR,'div.detailScore__wrapper').text.split('-')[1]
    dictionary['FT_Goals_A'] = int(away_goals_ft)

In [ ]:
def get_goal_minutes(id_jogo,dictionary,driver):
    def trata_minuto(minuto):
        minuto = minuto
        if "'" in minuto:
            minuto = minuto.replace("'","")
        if ":" in minuto:
            minuto = minuto.split(':')[0]
        if "+" in minuto:
            minuto = minuto.split('+')[0]
        return int(minuto)
    url = f'https://www.flashscore.com/match/{id_jogo}/#/match-summary/match-summary'
    if driver.current_url != url:
        driver.get(url)
    try:
        WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'div.smv__verticalSections')))
        soup = driver.page_source
        soup = BeautifulSoup(driver.page_source)
        min_goals_home = []
        min_goals_away = []
        try:
            home_incidents = soup.find_all('div',{"class":['smv__homeParticipant']})
            for i in home_incidents:
                if ('smv__incidentHomeScore' in str(i)) | ('footballOwnGoal-ico' in str(i)):
                    min_goals_home.append(trata_minuto(i.find('div',{"class":['smv__timeBox']}).text))
            dictionary['Goals_Minutes_Home'] = min_goals_home
        except:
            dictionary['Goals_Minutes_Home'] = min_goals_home
        try:
            away_incidents = soup.find_all('div',{"class":['smv__awayParticipant']})
            for i in away_incidents:
                if ('smv__incidentAwayScore' in str(i)) | ('footballOwnGoal-ico' in str(i)):
                    min_goals_away.append(trata_minuto(i.find('div',{"class":['smv__timeBox']}).text))
            dictionary['Goals_Minutes_Away'] = min_goals_away
        except:
                dictionary['Goals_Minutes_Away'] = min_goals_away
    except:
        pass   

### Jogos do Dia

In [ ]:
user_agent = UserAgent()
header = {'User-Agent': user_agent.random}

amanha = datetime.strftime(datetime.today() + timedelta(days=1),'%Y-%m-%d')
partidas_amanha = TinyDB(f'./FlashScore/{amanha}.json')

options = webdriver.ChromeOptions()
options.add_argument(f'user-agent={header["User-Agent"]}')
options.add_argument('--headless')

s = Service('./FlashScore/chromedriver.exe')
driver = webdriver.Chrome(service=s, options=options)
driver.maximize_window()

url = 'https://www.flashscore.com/'
driver.get(url)

try:
    WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')))
    button_cookies = driver.find_element(By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')
    button_cookies.click()
except:
    print("cookies already closed")

WebDriverWait(driver, 8).until(EC.element_to_be_clickable((By.CSS_SELECTOR,'.calendar__navigation--tomorrow')))
driver.execute_script("document.getElementsByClassName('otPlaceholder')[0].style.display='none';")
driver.find_element(By.CSS_SELECTOR,'.calendar__navigation--tomorrow').click()
sleep(5)

jogos = driver.find_elements(By.CSS_SELECTOR,'div.event__match--twoLine')
id_jogos = []
for i in jogos:
    id_jogos.append(i.get_attribute("id")[4:])

print(driver.title)
for id_jogo in tqdm(id_jogos):
    procura_duplicado = Query()
    if partidas_amanha.search(procura_duplicado.Id == id_jogo) == []:
        driver.get(f'https://www.flashscore.com/match/{id_jogo}/#/match-summary/match-summary')
        country = driver.find_element(By.CSS_SELECTOR,'span.tournamentHeader__country').text.split(':')[0]
        league = driver.find_element(By.CSS_SELECTOR,'span.tournamentHeader__country > a').text.split(' -')[0]
        if f'{country} - {league}' in leagues_list:
            try:
                jogo = {}

                get_main_header(id_jogo,jogo,driver)
                get_odds_ml_ft(id_jogo,jogo,driver)
                get_odds_ou_ht(id_jogo,jogo,driver)
                get_odds_ou_ft(id_jogo,jogo,driver)
                get_odds_btts_ft(id_jogo,jogo,driver)
                get_odds_ah_ft(id_jogo,jogo,driver)
                get_odds_cs_ft(id_jogo,jogo,driver)

                partidas_amanha.insert(jogo.copy())
            except:
                pass

driver.close()

In [ ]:
df = pd.DataFrame(partidas_amanha.all())

df = df.sort_values(['Time'])

df.reset_index(inplace=True, drop=True)
df.index = df.index.set_names(['Nº'])
df = df.rename(index=lambda x: x + 1)

df = df[['Id','Date','Time','League','Home','Away','Round',
         'FT_Odd_H','FT_Odd_D','FT_Odd_A',
         'HT_Odd_Over05','HT_Odd_Under05','FT_Odd_Over05','FT_Odd_Under05','FT_Odd_Over15','FT_Odd_Under15',
         'FT_Odd_Over25','FT_Odd_Under25',
         'Odd_BTTS_Yes','Odd_BTTS_No','Odd_AH_Neg05_H','Odd_AH_Pos05_A','Odd_AH_Pos05_H','Odd_AH_Neg05_A',
         'Odd_CS_0-0','Odd_CS_0-1','Odd_CS_0-2','Odd_CS_0-3',
         'Odd_CS_1-0','Odd_CS_1-1','Odd_CS_1-2','Odd_CS_1-3',
         'Odd_CS_2-0','Odd_CS_2-1','Odd_CS_2-2','Odd_CS_2-3',
         'Odd_CS_3-0','Odd_CS_3-1','Odd_CS_3-2','Odd_CS_3-3','Odd_CS_4-4']]
df.columns = ['Id','Date','Time','League','Home','Away','Round',
            'FT_Odd_H','FT_Odd_D','FT_Odd_A',
            'HT_Odd_Over05','HT_Odd_Under05','FT_Odd_Over05','FT_Odd_Under05','FT_Odd_Over15','FT_Odd_Under15',
            'FT_Odd_Over25','FT_Odd_Under25',
            'Odd_BTTS_Yes','Odd_BTTS_No','Odd_AH_Neg05_H','Odd_AH_Pos05_A','Odd_AH_Pos05_H','Odd_AH_Neg05_A',
            'CS_0x0','CS_0x1','CS_0x2','CS_0x3',
            'CS_1x0','CS_1x1','CS_1x2','CS_1x3',
            'CS_2x0','CS_2x1','CS_2x2','CS_2x3',
            'CS_3x0','CS_3x1','CS_3x2','CS_3x3','CS_4x4']

display(df)

### Bases de Dados

In [ ]:
links_2023_2024 = [
'https://www.flashscore.com/football/albania/superliga-2023-2024/results/',
'https://www.flashscore.com/football/andorra/primera-divisio-2023-2024/results/',
'https://www.flashscore.com/football/angola/girabola-2023-2024/results/',
'https://www.flashscore.com/football/armenia/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/australia/a-league-2023-2024/results/',
'https://www.flashscore.com/football/austria/bundesliga-2023-2024/results/',
'https://www.flashscore.com/football/austria/2-liga-2023-2024/results/',
'https://www.flashscore.com/football/azerbaijan/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/bahrain/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/bangladesh/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/belgium/jupiler-pro-league-2023-2024/results/',
'https://www.flashscore.com/football/belgium/challenger-pro-league-2023-2024/results/',
'https://www.flashscore.com/football/bosnia-and-herzegovina/premijer-liga-bih-2023-2024/results/',
'https://www.flashscore.com/football/bulgaria/parva-liga-2023-2024/results/',
'https://www.flashscore.com/football/cameroon/elite-one-2023-2024/results/',
'https://www.flashscore.com/football/costa-rica/primera-division-2023-2024/results/',
'https://www.flashscore.com/football/croatia/hnl-2023-2024/results/',
'https://www.flashscore.com/football/croatia/prva-nl-2023-2024/results/',
'https://www.flashscore.com/football/cyprus/first-division-2023-2024/results/',
'https://www.flashscore.com/football/czech-republic/fortuna-liga-2023-2024/results/',
'https://www.flashscore.com/football/denmark/superliga-2023-2024/results/',
'https://www.flashscore.com/football/denmark/1st-division-2023-2024/results/',
'https://www.flashscore.com/football/egypt/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/england/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/england/championship-2023-2024/results/',
'https://www.flashscore.com/football/england/league-one-2023-2024/results/',
'https://www.flashscore.com/football/england/league-two-2023-2024/results/',
'https://www.flashscore.com/football/england/national-league-2023-2024/results/',
'https://www.flashscore.com/football/england/fa-cup-2023-2024/results/',
'https://www.flashscore.com/football/england/efl-cup-2023-2024/results/',
'https://www.flashscore.com/football/england/premier-league-2-2023-2024/results/',
'https://www.flashscore.com/football/eswatini/swazi-mtn-premier-league-2023-2024/results/',
'https://www.flashscore.com/football/france/ligue-1-2023-2024/results/',
'https://www.flashscore.com/football/france/ligue-2-2023-2024/results/',
'https://www.flashscore.com/football/france/national-2023-2024/results/',
'https://www.flashscore.com/football/germany/bundesliga-2023-2024/results/',
'https://www.flashscore.com/football/germany/2-bundesliga-2023-2024/results/',
'https://www.flashscore.com/football/germany/3-liga-2023-2024/results/',
'https://www.flashscore.com/football/germany/dfb-pokal-2023-2024/results/',
'https://www.flashscore.com/football/ghana/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/greece/super-league-2023-2024/results/',
'https://www.flashscore.com/football/guatemala/liga-nacional-2023-2024/results/',
'https://www.flashscore.com/football/honduras/liga-nacional-2023-2024/results/',
'https://www.flashscore.com/football/hong-kong/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/hungary/otp-bank-liga-2023-2024/results/',
'https://www.flashscore.com/football/india/isl-2023-2024/results/',
'https://www.flashscore.com/football/india/i-league-2023-2024/results/',
'https://www.flashscore.com/football/indonesia/liga-1-2023-2024/results/',
'https://www.flashscore.com/football/iran/persian-gulf-pro-league-2023-2024/results/',
'https://www.flashscore.com/football/iraq/super-league-2023-2024/results/',
'https://www.flashscore.com/football/israel/ligat-ha-al-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-a-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-b-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-c-group-a-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-c-group-b-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-c-group-c-2023-2024/results/'
'https://www.flashscore.com/football/italy/serie-c-promotion-play-offs-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-c-play-out-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-a-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-b-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-c-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-d-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-e-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-f-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-g-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-h-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-group-i-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-play-offs-2023-2024/results/',
'https://www.flashscore.com/football/italy/serie-d-winners-stag-2023-2024/results/',
'https://www.flashscore.com/football/italy/coppa-italia-2023-2024/results/',
'https://www.flashscore.com/football/ivory-coast/ligue-1-2023-2024/results/',
'https://www.flashscore.com/football/jamaica/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/liechtenstein/liechtenstein-cup-2023-2024/results/',
'https://www.flashscore.com/football/luxembourg/national-division-2023-2024/results/',
'https://www.flashscore.com/football/malta/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/mexico/liga-mx-2023-2024/results/',
'https://www.flashscore.com/football/mexico/liga-de-expansion-mx-2023-2024/results/',
'https://www.flashscore.com/football/moldova/super-liga-2023-2024/results/',
'https://www.flashscore.com/football/mongolia/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/montenegro/prva-crnogorska-liga-2023-2024/results/',
'https://www.flashscore.com/football/morocco/botola-pro-2023-2024/results/',
'https://www.flashscore.com/football/netherlands/eredivisie-2023-2024/results/',
'https://www.flashscore.com/football/netherlands/eerste-divisie-2023-2024/results/',
'https://www.flashscore.com/football/netherlands/tweede-divisie-2023-2024/results/',
'https://www.flashscore.com/football/netherlands/derde-divisie-2023-2024/results/',
'https://www.flashscore.com/football/nicaragua/liga-primera-2023-2024/results/',
'https://www.flashscore.com/football/nigeria/npfl-2023-2024/results/',
'https://www.flashscore.com/football/northern-ireland/nifl-premiership-2023-2024/results/',
'https://www.flashscore.com/football/northern-ireland/nifl-championship-2023-2024/results/',
'https://www.flashscore.com/football/north-macedonia/1-mfl-2023-2024/results/',
'https://www.flashscore.com/football/poland/ekstraklasa-2023-2024/results/',
'https://www.flashscore.com/football/poland/division-1-2023-2024/results/',
'https://www.flashscore.com/football/portugal/liga-portugal-2023-2024/results/',
'https://www.flashscore.com/football/portugal/liga-portugal-2-2023-2024/results/',
'https://www.flashscore.com/football/portugal/taca-de-portugal-2023-2024/results/',
'https://www.flashscore.com/football/qatar/qsl-2023-2024/results/',
'https://www.flashscore.com/football/republic-of-the-congo/ligue-1-2023-2024/results/',
'https://www.flashscore.com/football/romania/liga-1-2023-2024/results/',
'https://www.flashscore.com/football/romania/liga-2-2023-2024/results/',
'https://www.flashscore.com/football/russia/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/san-marino/campionato-sammarinese-2023-2024/results/',
'https://www.flashscore.com/football/saudi-arabia/saudi-professional-league-2023-2024/results/',
'https://www.flashscore.com/football/scotland/premiership-2023-2024/results/',
'https://www.flashscore.com/football/scotland/championship-2023-2024/results/',
'https://www.flashscore.com/football/scotland/league-one-2023-2024/results/',
'https://www.flashscore.com/football/scotland/league-two-2023-2024/results/',
'https://www.flashscore.com/football/senegal/ligue-1-2023-2024/results/',
'https://www.flashscore.com/football/serbia/super-liga-2023-2024/results/',
'https://www.flashscore.com/football/serbia/prva-liga-2023-2024/results/',
'https://www.flashscore.com/football/slovakia/nike-liga-2023-2024/results/',
'https://www.flashscore.com/football/slovenia/prva-liga-2023-2024/results/',
'https://www.flashscore.com/football/spain/laliga-2023-2024/results/',
'https://www.flashscore.com/football/spain/laliga2-2023-2024/results/',
'https://www.flashscore.com/football/spain/primera-rfef-group-1-2023-2024/results/',
'https://www.flashscore.com/football/spain/primera-rfef-group-2-2023-2024/results/',
'https://www.flashscore.com/football/spain/primera-rfef-play-offs-2023-2024/results/',
'https://www.flashscore.com/football/spain/primera-rfef-promotion-play-offs-2023-2024/results/',
'https://www.flashscore.com/football/spain/segunda-rfef-group-1-2023-2024/results/',
'https://www.flashscore.com/football/spain/segunda-rfef-group-2-2023-2024/results/',
'https://www.flashscore.com/football/spain/segunda-rfef-group-3-2023-2024/results/',
'https://www.flashscore.com/football/spain/segunda-rfef-group-4-2023-2024/results/',
'https://www.flashscore.com/football/spain/segunda-rfef-group-5-2023-2024/results/',
'https://www.flashscore.com/football/spain/segunda-rfef-promotion-play-offs-2023-2024/results/',
'https://www.flashscore.com/football/spain/segunda-rfef-play-out-2023-2024/results/',
'https://www.flashscore.com/football/spain/copa-del-rey-2023-2024/results/',
'https://www.flashscore.com/football/sudan/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/switzerland/super-league-2023-2024/results/',
'https://www.flashscore.com/football/switzerland/challenge-league-2023-2024/results/',
'https://www.flashscore.com/football/thailand/thai-league-1-2023-2024/results/',
'https://www.flashscore.com/football/thailand/thai-league-2-2023-2024/results/',
'https://www.flashscore.com/football/tunisia/ligue-professionnelle-1-2023-2024/results/',
'https://www.flashscore.com/football/turkey/super-lig-2023-2024/results/',
'https://www.flashscore.com/football/turkey/1-lig-2023-2024/results/',
'https://www.flashscore.com/football/uganda/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/ukraine/premier-league-2023-2024/results/',
'https://www.flashscore.com/football/united-arab-emirates/uae-league-2023-2024/results/',
'https://www.flashscore.com/football/wales/cymru-premier-2023-2024/results/',
'https://www.flashscore.com/football/europe/champions-league-2023-2024/results/',
'https://www.flashscore.com/football/europe/europa-league-2023-2024/results/',
'https://www.flashscore.com/football/europe/europa-conference-league-2023-2024/results/']

In [ ]:
links_2024 = [
'https://www.flashscore.com/football/argentina/liga-profesional-2024/results/', 
'https://www.flashscore.com/football/argentina/copa-de-la-liga-profesional-2024/results/',
'https://www.flashscore.com/football/argentina/primera-nacional-2024/results/',
'https://www.flashscore.com/football/belarus/vysshaya-liga-2024/results/',
'https://www.flashscore.com/football/bolivia/division-profesional-2024/results/',
'https://www.flashscore.com/football/bolivia/nacional-b-2024/results/',
'https://www.flashscore.com/football/brazil/serie-a-2024/results/',
'https://www.flashscore.com/football/brazil/serie-b-2024/results/',
'https://www.flashscore.com/football/brazil/serie-c-2024/results/',
'https://www.flashscore.com/football/brazil/serie-d-2024/results/',
'https://www.flashscore.com/football/brazil/copa-do-brasil-2024/results/',
'https://www.flashscore.com/football/brazil/brasileiro-women-2024/results/',
'https://www.flashscore.com/football/brazil/brasileiro-u20-2024/results/',
'https://www.flashscore.com/football/brazil/brasileiro-u23-2024/results/',
'https://www.flashscore.com/football/canada/canadian-premier-league-2024/results/',
'https://www.flashscore.com/football/chile/primera-division-2024/results/',
'https://www.flashscore.com/football/chile/primera-b-2024/results/',
'https://www.flashscore.com/football/china/super-league-2024/results/',
'https://www.flashscore.com/football/colombia/primera-a-2024/results/',
'https://www.flashscore.com/football/colombia/primera-b-2024/results/',
'https://www.flashscore.com/football/dominican-republic/ldf-2024/results/',
'https://www.flashscore.com/football/ecuador/liga-pro-2024/results/',
'https://www.flashscore.com/football/ecuador/serie-b-2024/results/',
'https://www.flashscore.com/football/estonia/meistriliiga-2024/results/',
'https://www.flashscore.com/football/estonia/esiliiga-2024/results/',
'https://www.flashscore.com/football/fiji/premier-league-2024/results/',
'https://www.flashscore.com/football/finland/veikkausliiga-2024/results/',
'https://www.flashscore.com/football/finland/ykkonen-2024/results/',
'https://www.flashscore.com/football/georgia/crystalbet-erovnuli-liga-2024/results/',
'https://www.flashscore.com/football/iceland/besta-deild-karla-2024/results/',
'https://www.flashscore.com/football/ireland/premier-division-2024/results/',
'https://www.flashscore.com/football/ireland/division-1-2024/results/',
'https://www.flashscore.com/football/japan/j1-league-2024/results/',
'https://www.flashscore.com/football/japan/j2-league-2024/results/',
'https://www.flashscore.com/football/lithuania/a-lyga-2024/results/',
'https://www.flashscore.com/football/mozambique/mocambola-2024/results/',
'https://www.flashscore.com/football/norway/eliteserien-2024/results/',
'https://www.flashscore.com/football/norway/obos-ligaen-2024/results/',
'https://www.flashscore.com/football/panama/lpf-2024/results/',
'https://www.flashscore.com/football/paraguay/primera-division-2024/results/',
'https://www.flashscore.com/football/paraguay/division-intermedia-2024/results/',
'https://www.flashscore.com/football/peru/liga-1-2024/results/',
'https://www.flashscore.com/football/peru/liga-2-2024/results/',
'https://www.flashscore.com/football/singapore/premier-league-2024/results/',
'https://www.flashscore.com/football/south-africa/premier-league-2024/results/',
'https://www.flashscore.com/football/south-korea/k-league-1-2024/results/',
'https://www.flashscore.com/football/south-korea/k-league-2-2024/results/',
'https://www.flashscore.com/football/sweden/allsvenskan-2024/results/',
'https://www.flashscore.com/football/sweden/superettan-2024/results/',
'https://www.flashscore.com/football/uruguay/primera-division-2024/results/',
'https://www.flashscore.com/football/uruguay/segunda-division-2024/results/',
'https://www.flashscore.com/football/usa/mls-2024/results/',
'https://www.flashscore.com/football/usa/usl-championship-2024/results/',
'https://www.flashscore.com/football/uzbekistan/super-league-2024/results/',
'https://www.flashscore.com/football/venezuela/liga-futve-2024/results/',
'https://www.flashscore.com/football/venezuela/liga-futve-2-2024/results/',
'https://www.flashscore.com/football/vietnam/v-league-1-2024/results/',
'https://www.flashscore.com/football/south-america/copa-libertadores-2024/results/',
'https://www.flashscore.com/football/south-america/copa-sudamericana-2024/results/']

In [ ]:
for i in links_2023_2024:
    
    url = i
    colecao_partidas = TinyDB('./FlashScore/base_flashscore_2023_2024_new.json')
    
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')

    s = Service('./FlashScore/chromedriver.exe')
    driver = webdriver.Chrome(service=s, options=options)
    driver.maximize_window()

    
    try:
        driver.get(url)

        if ('No match found.' in driver.page_source) == False:
            try:
                WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')))
                button_cookies = driver.find_element(By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')
                button_cookies.click()
            except:
                print("cookies already closed")

            while('event__more' in driver.page_source):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")    
                button_more = driver.find_element(By.CSS_SELECTOR,'a.event__more.event__more--static')
                button_more.click()
                sleep(2)

            jogos = driver.find_elements(By.CSS_SELECTOR,'div.event__match--twoLine')
            id_jogos =[]
            for i in jogos:
                id_jogos.append(i.get_attribute("id")[4:])

            season = driver.find_element(By.CSS_SELECTOR,'div.heading__info').text

            print(driver.title)
            for id_jogo in tqdm(id_jogos):
                procura_duplicado = Query()
                if colecao_partidas.search(procura_duplicado.Id == id_jogo) == []:
                    try:
                        jogo = {}

                        get_main_header(id_jogo,jogo,driver)
                        insert_season(jogo,season)
                        get_score_ht_ft(id_jogo,jogo,driver)
                        get_odds_ml_ft(id_jogo,jogo,driver)
                        get_odds_ou_ht(id_jogo,jogo,driver)
                        get_odds_ou_ft(id_jogo,jogo,driver)
                        get_odds_btts_ft(id_jogo,jogo,driver)
                        get_odds_ah_ft(id_jogo,jogo,driver)
                        get_odds_cs_ft(id_jogo,jogo,driver)
                        get_goal_minutes(id_jogo,jogo,driver)

                        colecao_partidas.insert(jogo.copy())
                    except:
                        pass
        driver.close()
    except:
        pass

In [ ]:
for i in links_2024:
    
    url = i
    colecao_partidas = TinyDB('./FlashScore/base_flashscore_2024.json')
    
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')

    s = Service('./FlashScore/chromedriver.exe')
    driver = webdriver.Chrome(service=s, options=options)
    driver.maximize_window()
    
    try:
        driver.get(url)

        if ('No match found.' in driver.page_source) == False:
            try:
                WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')))
                button_cookies = driver.find_element(By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')
                button_cookies.click()
            except:
                print("cookies already closed")

            while('event__more' in driver.page_source):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")    
                button_more = driver.find_element(By.CSS_SELECTOR,'a.event__more.event__more--static')
                button_more.click()
                sleep(2)

            jogos = driver.find_elements(By.CSS_SELECTOR,'div.event__match--twoLine')
            id_jogos =[]
            for i in jogos:
                id_jogos.append(i.get_attribute("id")[4:])

            season = driver.find_element(By.CSS_SELECTOR,'div.heading__info').text

            print(driver.title)
            for id_jogo in tqdm(id_jogos):
                procura_duplicado = Query()
                if colecao_partidas.search(procura_duplicado.Id == id_jogo) == []:
                    try:
                        jogo = {}

                        get_main_header(id_jogo,jogo,driver)
                        insert_season(jogo,season)
                        get_score_ht_ft(id_jogo,jogo,driver)
                        get_odds_ml_ft(id_jogo,jogo,driver)
                        get_odds_ou_ht(id_jogo,jogo,driver)
                        get_odds_ou_ft(id_jogo,jogo,driver)
                        get_odds_btts_ft(id_jogo,jogo,driver)
                        get_odds_ah_ft(id_jogo,jogo,driver)
                        get_odds_cs_ft(id_jogo,jogo,driver)
                        get_goal_minutes(id_jogo,jogo,driver)

                        colecao_partidas.insert(jogo.copy())
                    except:
                        pass
        driver.close()
    except:
        pass

In [5]:
colecao_partidas1 = TinyDB('./FlashScore/base_flashscore_2024_new.json')
colecao_partidas2 = TinyDB('./FlashScore/base_flashscore_2023_2024_new.json')

df1 = pd.DataFrame(colecao_partidas1.all())
df2 = pd.DataFrame(colecao_partidas2.all())

df = pd.concat([df1,df2])
df.drop_duplicates(subset=['Id'], inplace=True)

df[['Dia','Mes','Ano']] = df['Date'].str.split('/',expand=True)
df = df.drop(['Date'], axis=1)
df['Date'] = df['Ano']+'-'+df['Mes']+'-'+ df['Dia']
df = df.drop(columns=['Dia','Mes','Ano'], axis=1)

df = df[['Date','Time','League','Season','Round','Home','Away','HT_Goals_H','HT_Goals_A','FT_Goals_H','FT_Goals_A',
'FT_Odd_H','FT_Odd_D','FT_Odd_A',
'HT_Odd_Over05','HT_Odd_Under05','FT_Odd_Over05','FT_Odd_Under05','FT_Odd_Over15','FT_Odd_Under15',
'FT_Odd_Over25','FT_Odd_Under25',
'Odd_BTTS_Yes','Odd_BTTS_No',
'Odd_CS_0-0','Odd_CS_0-1','Odd_CS_0-2','Odd_CS_0-3',
'Odd_CS_1-0','Odd_CS_1-1','Odd_CS_1-2','Odd_CS_1-3',
'Odd_CS_2-0','Odd_CS_2-1','Odd_CS_2-2','Odd_CS_2-3',
'Odd_CS_3-0','Odd_CS_3-1','Odd_CS_3-2','Odd_CS_3-3','Odd_CS_4-4',
'Goals_Minutes_Home','Goals_Minutes_Away']]

df.columns = ['Date','Time','League','Season','Round','Home','Away','HT_Goals_H','HT_Goals_A','FT_Goals_H','FT_Goals_A',
'FT_Odd_H','FT_Odd_D','FT_Odd_A',
'HT_Odd_Over05','HT_Odd_Under05','FT_Odd_Over05','FT_Odd_Under05','FT_Odd_Over15','FT_Odd_Under15',
'FT_Odd_Over25','FT_Odd_Under25',
'Odd_BTTS_Yes','Odd_BTTS_No',
'CS_0x0','CS_0x1','CS_0x2','CS_0x3',
'CS_1x0','CS_1x1','CS_1x2','CS_1x3',
'CS_2x0','CS_2x1','CS_2x2','CS_2x3',
'CS_3x0','CS_3x1','CS_3x2','CS_3x3','CS_4x4',
'Goals_Minutes_Home','Goals_Minutes_Away']

df.fillna(0, inplace=True)

df = df.sort_values(['Date','Time'])

df = df.dropna()
df = df.reset_index(drop=True)
df.index += 1

df

# df.to_excel('Base_de_Dados_FlashScore.xlsx',index=False)

,Date,Time,League,Season,Round,Home,Away,HT_Goals_H,HT_Goals_A,FT_Goals_H,FT_Goals_A,FT_Odd_H,FT_Odd_D,FT_Odd_A,HT_Odd_Over05,HT_Odd_Under05,FT_Odd_Over05,FT_Odd_Under05,FT_Odd_Over15,FT_Odd_Under15,FT_Odd_Over25,FT_Odd_Under25,Odd_BTTS_Yes,Odd_BTTS_No,CS_0x0,CS_0x1,CS_0x2,CS_0x3,CS_1x0,CS_1x1,CS_1x2,CS_1x3,CS_2x0,CS_2x1,CS_2x2,CS_2x3,CS_3x0,CS_3x1,CS_3x2,CS_3x3,CS_4x4,Goals_Minutes_Home,Goals_Minutes_Away
1,2024-04-03,13:00,ALBANIA - SUPERLIGA,2023/2024,ROUND 27,Dinamo Tirana,Teuta,0,0,1,0,2.02,3.2,3.3,1.44,2.32,1.04,8.5,1.34,2.88,2.05,1.62,1.83,1.93,8.25,7.4,13.0,32.0,5.6,5.5,9.75,25.0,7.7,7.5,13.0,35.0,14.5,14.5,25.0,65.0,500.0,[90],[]


### Bases FutPythonTrader

In [7]:
import pandas as pd
from tinydb import TinyDB

colecao_partidas1 = TinyDB('./FlashScore/base_flashscore_2019.json')
colecao_partidas2 = TinyDB('./FlashScore/base_flashscore_2020.json')
colecao_partidas3 = TinyDB('./FlashScore/base_flashscore_2021.json')
colecao_partidas4 = TinyDB('./FlashScore/base_flashscore_2022.json')
colecao_partidas5 = TinyDB('./FlashScore/base_flashscore_2023.json')
colecao_partidas6 = TinyDB('./FlashScore/base_flashscore_2024.json')
colecao_partidas7 = TinyDB('./FlashScore/base_flashscore_2021_2022.json')
colecao_partidas8 = TinyDB('./FlashScore/base_flashscore_2022_2023.json')
colecao_partidas9 = TinyDB('./FlashScore/base_flashscore_2023_2024.json')

df1 = pd.DataFrame(colecao_partidas1.all())
df2 = pd.DataFrame(colecao_partidas2.all())
df3 = pd.DataFrame(colecao_partidas3.all())
df4 = pd.DataFrame(colecao_partidas4.all())
df5 = pd.DataFrame(colecao_partidas5.all())
df6 = pd.DataFrame(colecao_partidas6.all())
df7 = pd.DataFrame(colecao_partidas7.all())
df8 = pd.DataFrame(colecao_partidas8.all())
df9 = pd.DataFrame(colecao_partidas9.all())

df = pd.concat([df1,df2,df3,df4,df5,df6,df7,df8,df9])
df.drop_duplicates(subset=['Id'], inplace=True)

df[['Dia','Mes','Ano']] = df['Date'].str.split('/',expand=True)
df = df.drop(['Date'], axis=1)
df['Date'] = df['Ano']+'-'+df['Mes']+'-'+ df['Dia']
df = df.drop(columns=['Dia','Mes','Ano'], axis=1)

df = df[['Id','Date','Time','League','Home','Away','Round','Season','Goals_H_HT','Goals_A_HT','Goals_H','Goals_A',
          'Odd_H','Odd_D','Odd_A','Odd_Over25','Odd_Under25','Odd_BTTS_Yes','Odd_BTTS_No','Goals_Minutes_Home','Goals_Minutes_Away']]

df.fillna(0, inplace=True)

df = df.sort_values(['Date','Time'])

df = df.dropna()
df = df.reset_index(drop=True)
df.index += 1

display(df.head())

,Id,Date,Time,League,Home,Away,Round,Season,Goals_H_HT,Goals_A_HT,Goals_H,Goals_A,Odd_H,Odd_D,Odd_A,Odd_Over25,Odd_Under25,Odd_BTTS_Yes,Odd_BTTS_No,Goals_Minutes_Home,Goals_Minutes_Away
1,SMWRIDUs,2019-01-19,16:00,BOLIVIA - DIVISION PROFESIONAL,Sport Boys,Always Ready,APERTURA,2019,1.0,0.0,1,1,1.61,4.20,4.50,1.47,2.60,1.50,2.50,[45],[65]
2,ddyWHXql,2019-01-19,18:15,BOLIVIA - DIVISION PROFESIONAL,Real Potosi,Guabira,APERTURA,2019,1.0,1.0,3,1,1.57,3.80,5.00,1.53,2.40,1.57,2.25,"[20, 49, 82]",[37]
3,zmzzHiaf,2019-01-19,20:30,BOLIVIA - DIVISION PROFESIONAL,The Strongest,Aurora,APERTURA,2019,1.0,0.0,2,1,1.18,6.50,10.00,1.30,3.50,1.72,2.00,"[15, 83]",[69]
4,IBPvGBE0,2019-01-20,16:00,BOLIVIA - DIVISION PROFESIONAL,Destroyers,Blooming,APERTURA,2019,1.0,0.0,1,0,2.87,3.30,2.25,1.85,1.95,1.66,2.10,[45],[]
5,QyTnEkqD,2019-01-20,18:15,BOLIVIA - DIVISION PROFESIONAL,San Jose,Nacional Potosi,APERTURA,2019,1.0,1.0,2,2,1.44,4.75,5.25,1.33,3.40,1.44,2.62,"[30, 88]","[37, 82]"


In [8]:
df.to_excel('./FlashScore/Base_de_Dados_FlashScore.xlsx',index=False)

In [9]:
len(df)

109837